## Logistic Regression Pipeline Overview / 逻辑回归建模流程概览

### English

To ensure a structured and reproducible modeling process, a complete Logistic Regression pipeline was implemented. The workflow follows standard supervised learning procedures and incorporates insights from exploratory data analysis (EDA).

The modeling process consists of the following steps:

1. **Load the cleaned dataset**
   Import the preprocessed dataset after data cleaning.

2. **Feature selection**
   Select the most relevant numerical features based on correlation analysis, and include important categorical variables identified during EDA.

3. **Categorical encoding**
   Convert categorical variables (e.g., Color) into numerical format using one-hot encoding.

4. **Train-test split**
   Split the dataset into training and testing sets while preserving class distribution using stratified sampling.

5. **Feature scaling (standardization)**
   Standardize numerical features to ensure comparable scales, which is essential for Logistic Regression.

6. **Model training**
   Train a Logistic Regression model using the training dataset.

7. **Prediction**
   Generate predictions on the test dataset.

8. **Model evaluation**
   Evaluate model performance using metrics such as accuracy, F1-score, and classification report.

9. **Cross-validation**
   Apply cross-validation to assess model robustness and reduce the risk of overfitting.

10. **Model interpretation (optional)**
    Analyze model coefficients to understand the contribution of each feature.

This pipeline ensures a systematic approach to model development, from data preparation to evaluation and interpretation.

---

### 中文

为了保证建模过程的规范性与可复现性，本研究构建了完整的逻辑回归（Logistic Regression）建模流程。该流程遵循标准的监督学习步骤，并结合前期探索性数据分析（EDA）的结果进行设计。

具体建模步骤如下：

1. **数据导入**
   读取清洗后的数据集。

2. **特征选择**
   基于相关性分析选择最重要的数值特征，并加入EDA中识别出的关键类别变量。

3. **类别变量编码**
   对类别变量（如 Color）进行独热编码（One-hot encoding），转换为数值形式。

4. **训练集与测试集划分**
   使用分层抽样（stratified sampling）划分数据集，保证类别比例一致。

5. **特征标准化**
   对数值特征进行标准化处理，使各变量具有可比尺度，这是逻辑回归的重要前提。

6. **模型训练**
   使用训练集拟合逻辑回归模型。

7. **模型预测**
   在测试集上进行预测。

8. **模型评估**
   使用准确率（Accuracy）、F1-score 和分类报告等指标评估模型性能。

9. **交叉验证**
   通过交叉验证评估模型稳定性，降低过拟合风险。

10. **模型解释**
    分析模型系数，理解各特征对预测结果的影响。

该流程从数据准备到模型评估形成完整闭环，有助于提高模型的可靠性与解释性。


### Step 0： Import libraries

In [45]:
#Basic libraries
import pandas as pd
import numpy as np

#ML libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn import set_config

#Evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

### Step 1：Load dataset

In [35]:
# Load cleaned dataset
df = pd.read_csv("../data/processed/water_quality_cleaned.csv")

# Check basic info
print(df.shape)
df.head()

(734756, 23)


,pH,Iron,Nitrate,Chloride,Lead,Zinc,Color,Turbidity,Fluoride,Copper,...,Chlorine,Manganese,Total Dissolved Solids,Source,Water Temperature,Air Temperature,Month,Day,Time of Day,Target
0,6.917863,8.050000e-05,3.734167,227.029851,7.850000e-94,1.245317,Faint Yellow,0.019007,0.622874,0.437835,...,3.292038,8.020000e-07,284.641984,Lake,15.348981,71.220586,11.0,26.0,16.0,0
1,5.443762,2.010586e-02,3.816994,230.995630,5.290000e-76,0.528280,Light Yellow,0.319956,0.423423,0.431588,...,3.560224,7.007989e-02,570.054094,River,11.643467,44.891330,1.0,31.0,8.0,0
2,8.091909,2.167128e-03,9.925788,186.540872,4.170000e-132,3.807511,Light Yellow,0.004867,0.222912,0.616574,...,3.177849,3.296139e-03,168.075545,Spring,15.249416,69.336671,6.0,29.0,7.0,0
3,7.258203,6.110000e-09,9.261676,182.242341,4.400000e-224,0.416478,Colorless,0.047803,1.016196,0.298093,...,2.325094,6.020000e-16,214.553104,River,15.891905,61.139140,4.0,11.0,4.0,0
4,7.100650,3.019038e-03,3.620641,157.043934,6.460000e-148,0.112994,Colorless,0.050613,0.842107,0.391602,...,2.284971,8.840000e-07,113.909077,River,11.899376,14.010268,4.0,7.0,12.0,0


### Step 2：Feature selection



特征选择基于探索性数据分析（EDA）的结果，尤其是各变量与目标变量之间的相关性。

数值特征中，选择了与目标变量绝对相关性较高（约大于 0.13）的前10个变量，包括 Turbidity、Copper、Chloride、Manganese、Iron、Fluoride、Nitrate、Odor、Chlorine 和 Sulfate。这些变量被认为具有更强的预测能力，更有助于区分安全与不安全水样。

相反，对于相关性极低的变量（如时间变量、温度、Conductivity 等），将其剔除，以减少噪声并提升模型表现。

此外，类别变量 "Color" 被保留，因为EDA分析表明不同颜色对应的水质不安全比例存在显著差异，说明该变量具有较强的判别信息。

该特征选择方法有助于简化模型结构，提高模型可解释性，并增强模型的泛化能力。

Feature selection was performed based on the results of exploratory data analysis (EDA), particularly the correlation between each feature and the target variable.

The top numerical features were selected according to their relatively high absolute correlation values with the target (above approximately 0.13). These features include turbidity, copper, chloride, manganese, iron, fluoride, nitrate, odor, chlorine, and sulfate. These variables are considered to have stronger predictive power and are more relevant for distinguishing between safe and unsafe water samples.

In contrast, features with negligible correlation (such as temporal variables, temperature, and conductivity) were excluded to reduce noise and improve model performance.

Additionally, the categorical feature "Color" was included due to its strong observed relationship with the target variable. EDA showed that different color categories correspond to significantly different unsafe rates, indicating that this feature provides meaningful information for classification.

This feature selection approach helps simplify the model, improve interpretability, and enhance generalization performance.





In [36]:
# Select top numerical features
selected_features = [
    "Turbidity", "Copper", "Chloride", "Manganese",
    "Iron", "Fluoride", "Nitrate", "Odor",
    "Chlorine", "Sulfate"
]

# Add important categorical feature
selected_features += ["Color"]

# Define features and target
X = df[selected_features]
y = df["Target"]

### Step 3：Encode categorical variables

One-hot encoding for categorical variable (Color)

In [37]:
X = pd.get_dummies(X, columns=["Color"], drop_first=True)

### Step 4：Train-test split

In [38]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   #  Keep class balance
)

### Step 5：Feature scaling

Before training the model, feature scaling is applied to ensure that all numerical variables are on a comparable scale. This is important for Logistic Regression, as it relies on distance-based optimization and can be sensitive to differences in feature magnitude.

In [39]:
# Standardization
scaler = StandardScaler()

# Fit on training data
X_train = scaler.fit_transform(X_train)

# Transform test data
X_test = scaler.transform(X_test)

### Step 6：Train Logistic model

In [40]:
#Initialize Logistic Regression
set_config(display='text')      #controller output format

model = LogisticRegression(class_weight='balanced', max_iter=200)
# - class_weight='balanced': handles class imbalance by giving more weight to the minority class
# - max_iter=200: increases the number of iterations to ensure convergence
model.fit(X_train, y_train)
print(model)


LogisticRegression(class_weight='balanced', max_iter=200)


### Step 7： Prediction

In [41]:
# Predict on test set
y_pred = model.predict(X_test)

### Step 8：Evaluate model

The performance of the Logistic Regression model was evaluated by comparing the predicted labels with the true labels in the test dataset.

Several evaluation metrics were used:

Accuracy measures the overall proportion of correctly classified samples.

F1-score provides a balance between precision and recall, which is especially important for imbalanced datasets.

Classification report gives a detailed breakdown of precision, recall, and F1-score for each class.

Confusion matrix summarizes prediction outcomes and helps identify false positives and false negatives.

These metrics provide a comprehensive understanding of the model's performance and its ability to correctly identify unsafe water samples.

通过将测试数据集中模型预测的标签与真实标签进行比较，对逻辑回归模型的性能进行了评估。

本步骤使用了多种评估指标：

准确率（Accuracy）：衡量被正确分类的样本在总体中的比例。

F1 分数（F1-score）：在精确率（Precision）和召回率（Recall）之间取得平衡，尤其适用于类别不平衡的数据集。

分类报告（Classification report）：详细展示每个类别的精确率、召回率和 F1 分数。

混淆矩阵（Confusion matrix）：总结预测结果，有助于识别假阳性（False Positive）和假阴性（False Negative）。

这些指标能够全面反映模型的性能，以及其正确识别不安全水样的能力。

In [42]:
# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# F1-score
print("F1 Score:", f1_score(y_test, y_pred))

#Classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.814000489955904
F1 Score: 0.64816507266338

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.83      0.87    113127
           1       0.57      0.74      0.65     33825

    accuracy                           0.81    146952
   macro avg       0.75      0.79      0.76    146952
weighted avg       0.84      0.81      0.82    146952


Confusion Matrix:
 [[94442 18685]
 [ 8648 25177]]



模型性能分析

逻辑回归模型的准确率为 0.81，但由于数据存在类别不平衡问题，仅依赖准确率并不能全面反映模型性能。因此，本分析主要采用 F1-score（0.65）作为核心评价指标。

模型在识别安全水样（类别 0）方面表现较好，具有较高的精确率（0.92）和 F1-score（0.87）。对于不安全水样（类别 1），模型的召回率达到 0.74，说明大部分不安全样本能够被成功识别。然而，该类别的精确率相对较低（0.57），表明存在一定数量的误报（即将安全样本误判为不安全）。

混淆矩阵进一步表明，尽管模型能够捕捉到大部分不安全样本，但仍存在一定程度的误分类，主要体现在假阳性和假阴性上。

总体而言，该模型在识别不安全水样方面具备一定能力，但仍有提升空间，尤其是在保持较高召回率的同时，降低误报率方面。

Model Performance Analysis

The Logistic Regression model achieved an accuracy of 0.81; however, due to class imbalance, accuracy alone is not a reliable indicator of performance. Therefore, the F1-score (0.65) was used as the primary evaluation metric.

The model performs well in identifying safe water samples (class 0), with high precision (0.92) and F1-score (0.87). For unsafe water samples (class 1), the model achieves a recall of 0.74, indicating that most unsafe cases are successfully detected. However, the precision for this class is relatively low (0.57), suggesting the presence of false positives.

The confusion matrix further confirms that while the model captures a large portion of unsafe cases, some misclassification still occurs, particularly in the form of false positives and false negatives.

Overall, the model demonstrates a reasonable ability to identify unsafe water, but there is room for improvement, especially in reducing false alarms while maintaining high recall.

### Step 9： Cross-validation

In [43]:
#  Cross-validation
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")

print("Cross-validation F1 scores:", cv_scores)
print("Mean CV F1:", cv_scores.mean())

Cross-validation F1 scores: [0.64682011 0.64767298 0.64955778 0.64907686 0.64684277]
Mean CV F1: 0.6479940998656956


交叉验证分析

为了评估模型的稳健性，使用训练数据集进行了5折交叉验证。五个折中的F1分数大约在0.646到0.649之间，平均值为0.648。

交叉验证分数之间的低方差表明模型具有良好的稳定性，对不同的数据划分不敏感。此外，平均交叉验证分数与测试集表现一致，说明模型不存在过拟合问题。

然而，整体F1分数仍处于中等水平，这表明虽然模型具有一定的可靠性，但其预测性能仍然有限。这也促使我们在后续步骤中探索更复杂的模型。

Cross-validation Analysis

To assess the robustness of the model, 5-fold cross-validation was performed using the training dataset. The F1-scores across the five folds ranged from approximately 0.646 to 0.649, with a mean value of 0.648.

The low variance among the cross-validation scores indicates that the model is stable and not sensitive to different data splits. Furthermore, the mean cross-validation score is consistent with the test set performance, suggesting that the model does not suffer from overfitting.

However, the overall F1-score remains moderate, indicating that while the model is reliable, its predictive performance is limited. This motivates the exploration of more complex models in subsequent steps.

### Step 10： Model Interpretation

In [44]:
feature_names = X.columns

coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", key=abs, ascending=False)

print(coefficients)

                 Feature  Coefficient
13          Color_Yellow     0.498455
3              Manganese     0.462544
0              Turbidity     0.443409
1                 Copper     0.388186
2               Chloride     0.361569
4                   Iron     0.360734
11    Color_Light Yellow     0.360413
7                   Odor     0.290208
5               Fluoride     0.289734
6                Nitrate     0.287005
8               Chlorine     0.248981
10    Color_Faint Yellow     0.228888
9                Sulfate     0.217334
12  Color_Near Colorless    -0.006713


模型解释 / Model Interpretation

为了对逻辑回归模型进行解释，我们分析了各个特征的系数。这些系数反映了每个特征对样本被判定为不安全水质概率的影响方向和强度。

系数为正的特征会增加被判定为不安全水质的可能性，而系数为负则表示具有一定的保护作用。

结果表明，与颜色相关的特征（尤其是黄色）、浑浊度，以及锰、铜、氯化物和铁等金属相关变量，是预测水质不安全的最重要因素。这一结果与探索性数据分析中的发现高度一致。

总体而言，该模型验证了水的外观（颜色和浑浊度）以及化学污染是判断水质的重要指标。


### Model Interpretation

To interpret the Logistic Regression model, feature coefficients were analyzed. These coefficients indicate the direction and strength of each feature's influence on the probability of a sample being classified as unsafe water.

Features with positive coefficients increase the likelihood of unsafe classification, while negative coefficients indicate a protective effect.

The results show that color-related features (especially yellow), turbidity, and metal-related variables such as manganese, copper, chloride, and iron are the strongest predictors of unsafe water. This aligns well with the findings from the exploratory data analysis.

Overall, the model confirms that water appearance (color and turbidity) and chemical contamination are key indicators of water quality.


### Step 11：Final Conclusion
逻辑回归模型提供了一个稳定且具有可解释性的基准，其 F1 分数约为 0.65。尽管该模型成功捕捉到了浊度、色度和金属浓度等关键预测因子，但其整体表现仍处于中等水平。因此，将探索更先进的模型以提高预测准确性。

局限性：
- 线性模型：无法捕捉非线性关系
- 性能一般（F1 ≈ 0.65）
- 对特征工程敏感

下一步工作的动因：
鉴于逻辑回归（Logistic Regression）的性能有限，将探索神经网络（Neural Networks）和集成方法（Ensemble methods）等更复杂的模型。

The Logistic Regression model provides a stable and interpretable baseline, achieving an F1-score of approximately 0.65. While it successfully captures key predictors such as turbidity, color, and metal concentrations, its overall performance remains moderate. Therefore, more advanced models will be explored to improve predictive accuracy.

Limitations:
- Linear model: cannot capture non-linear relationships
- Moderate performance (F1 ≈ 0.65)
- Sensitive to feature engineering

Motivation for next step:
Due to the limited performance of Logistic Regression, more complex models such as Neural Networks and Ensemble methods will be explored.

